# Candidate Generation (Combined): Rolling Popularity + Content Text/Entity

This notebook is **self-contained**: it rebuilds both baseline channels from the shared project data instead of reading the saved baseline recommendation artifacts. It only depends on the stage outputs that every model in this project uses:

- Stage 1: `candidates_<split>.parquet`, `impressions_<split>.parquet` and `news_catalogue.parquet`.
- Stage 3: `article_index.parquet`, `article_availability.parquet`, `text64.npy` and `article_entity100.npy`.
- Stage 4: `request_profiles_<split>/*.parquet` with `full_history_rows`.

It combines two non-neural channels into one standard candidate table:

- **Popular channel:** the first 200 articles from the rolling prior-hour popularity order (`prior_hour_clicks`, tie-broken by prior-hour exposures and `news_code`), after removing every article in the request's causal history.
- **Content channel:** the next 250 articles from the text/entity cosine baseline (`content_text_entity`), after removing history articles and any article already selected by the popular channel.

The result is a **450-article list per request**: popularity first (`cutoff = 1..200`) and content second (`cutoff = 201..450`).

Protocol:

- Target split: `validation`.
- Popularity uses only completed hours strictly before the request hour (prequential, no leakage).
- Eligibility is `available_from <= timestamp`.
- All `full_history_rows` are removed from both channels.
- Content ties are broken by training popularity descending, then `news_code` ascending.
- Empty/zero content profiles fall back to the global popularity order, still excluding history and popular articles.
- The combined table uses the shared long format and is evaluated with `cg_evaluation.py`.

## 0. Configure

**This is the only cell you need to edit.** Set `REUSE_ARTIFACTS = False` to retrain the model
and rebuild every artifact. Candidate lists are always regenerated for `TARGET_SPLIT`, so a new
validation or test split is served normally.


In [ ]:
# ---- Where to run ----------------------------------------------------------------------------
# "colab" = Google Colab with Google Drive, "local" = this computer.
ENVIRONMENT = "colab"  # "local" or "colab"

# ---- Paths -------------------------------------------------------------------------------------
# Data path: the prepared data. It must contain
# 1_Data_Preparation/processed/, 3_Representations/artifacts/ and
# 4_Feature_Engineering/artifacts/prequential-all/.
# On Colab, the zipped data in <submission>/2.Feature_Engineering is unpacked into the data
# path on the first run (a few minutes); later runs reuse it.
COLAB_DATA_PATH = None  # used on Colab; None = the project folder (4.Code)
LOCAL_DATA_PATH = None  # used locally; None = the project folder

# ---- Experiment ------------------------------------------------------------------------------
# Split to evaluate: "test" for the final results, or "validation".
TARGET_SPLIT = "validation"

# True: load the saved model checkpoint and skip training. Serving still runs, so candidate
# lists are always regenerated for TARGET_SPLIT.
# False: retrain the model from scratch.
# Baselines without a model reuse any cached intermediate state, but their candidate lists are
# still regenerated.
REUSE_ARTIFACTS = True

# ---- Reused artifacts when REUSE_ARTIFACTS=True -----------------------------------------------
# No trained model or cached intermediate state: the combined baseline has no artifact to reuse.
# Candidate lists are always regenerated.


## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl


if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    CODE_ROOT = Path("/content/drive/MyDrive/submission/4.Code")
else:
    CODE_ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
    )
ROOT = CODE_ROOT
DATA_PATH = COLAB_DATA_PATH if ENVIRONMENT == "colab" else LOCAL_DATA_PATH
DATA_ROOT = (
    Path(
        DATA_PATH
        or os.environ.get("NEWSREC_DATA_ROOT")
        or os.environ.get("NEWSREC_ROOT")
        or CODE_ROOT
    )
    .expanduser()
    .resolve()
)
REQUIRED_DATA = [
    "1_Data_Preparation/processed",
    "3_Representations/artifacts",
    "4_Feature_Engineering/artifacts/prequential-all",
]
SUBMISSION_ROOT = CODE_ROOT.parent if ENVIRONMENT == "colab" else None
BUNDLE_DIR = SUBMISSION_ROOT / "2.Feature_Engineering" if SUBMISSION_ROOT else None
if BUNDLE_DIR is not None and BUNDLE_DIR.is_dir():
    import zipfile

    for archive in sorted(BUNDLE_DIR.glob("*.zip")):
        marker = DATA_ROOT / f".unpacked_{archive.stem}"
        if not marker.exists():
            print(f"Unpacking {archive.name} into {DATA_ROOT} (first run only) ...")
            with zipfile.ZipFile(archive) as bundle:
                bundle.extractall(DATA_ROOT)
            marker.touch()

missing_data = [
    str(DATA_ROOT / sub) for sub in REQUIRED_DATA if not (DATA_ROOT / sub).exists()
]
if missing_data:
    raise FileNotFoundError(
        f"{ENVIRONMENT.upper()}_DATA_PATH is missing these folders:\n"
        + "\n".join(missing_data)
    )
REPLAY_ROOT = DATA_ROOT
sys.path.insert(0, str(CODE_ROOT / "5_Models" / "5_01_Candidate_Generation"))

from cg_evaluation import CandidateEvaluator, wide_table
from cg_result_writer import CandidateWriter

PROCESSED = DATA_ROOT / "1_Data_Preparation" / "processed"
FEATURES = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"
ARTIFACTS = (
    CODE_ROOT.parent / "3.Model_Artifacts" / "CG"
    if ENVIRONMENT == "colab"
    else DATA_ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
)
RESULTS = DATA_ROOT / "results"

POPULAR_TOP_N = 200
CONTENT_TOP_N = 250
CANDIDATE_CAP = POPULAR_TOP_N + CONTENT_TOP_N
M_VALUES = (450,)
MODEL_NAME = "baseline_content_popularity_combined"
HOUR_NS = 3_600 * 10**9

if TARGET_SPLIT not in {"validation", "test"}:
    raise ValueError("TARGET_SPLIT must be validation or test")
if CANDIDATE_CAP < max(M_VALUES):
    raise ValueError("CANDIDATE_CAP must cover the largest M")

OUTPUT_DIR = ARTIFACTS / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"

REQUIRED_INPUTS = [
    PROCESSED / "candidates_train.parquet",
    PROCESSED / f"impressions_{TARGET_SPLIT}.parquet",
    PROCESSED / "news_catalogue.parquet",
    REPRESENTATIONS / "article_index.parquet",
    REPRESENTATIONS / "article_availability.parquet",
    REPRESENTATIONS / "text64.npy",
    REPRESENTATIONS / "article_entity100.npy",
]
for path in REQUIRED_INPUTS:
    if not path.exists():
        raise FileNotFoundError(f"Missing required project input: {path}")

PROFILE_DIR = FEATURES / f"request_profiles_{TARGET_SPLIT}"
if not sorted(PROFILE_DIR.glob("*.parquet")):
    raise FileNotFoundError(f"No request profiles under {PROFILE_DIR}")

print("Repository:", ROOT)
print("Target split:", TARGET_SPLIT)
print(
    f"Combined list: {POPULAR_TOP_N} popular + {CONTENT_TOP_N} content = {CANDIDATE_CAP}"
)
print("Output:", OUTPUT_PATH)

## 2. Catalogue, availability and request profiles

The article index uses `article_row = 1..N`, with row `0` reserved for padding. Availability is mapped to article rows so the popularity and content channels share one eligibility rule. Requests come from the stage 4 profiles, which also supply the causal history.

In [2]:
index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code"],
).sort("article_row")
if not np.array_equal(index["article_row"].to_numpy(), np.arange(1, index.height + 1)):
    raise ValueError("article_index rows must be 1..N with no gaps")

n_items = index.height + 1
rows = index["article_row"].to_numpy().astype(np.int64)
codes = index["news_code"].to_numpy().astype(np.int64)
row_of_code = np.zeros(int(codes.max()) + 1, dtype=np.int64)
row_of_code[codes] = rows
row_to_code = np.zeros(n_items, dtype=np.int32)
row_to_code[rows] = codes.astype(np.int32)

availability = pl.read_parquet(
    REPRESENTATIONS / "article_availability.parquet",
    columns=["news_code", "available_from"],
)
meta = index.join(availability, on="news_code", how="left")

avail_row = np.full(n_items, np.iinfo(np.int64).max, dtype=np.int64)
avail_row[rows] = meta["available_from"].dt.epoch("ns").to_numpy().astype(np.int64)


# Reference hour: midnight of the first training day.
first_ts = (
    pl.scan_parquet(PROCESSED / "candidates_train.parquet")
    .select(pl.col("timestamp").min())
    .collect()
    .item()
)
T0_NS = int(
    np.datetime64(first_ts, "ns")
    .astype("datetime64[D]")
    .astype("datetime64[ns]")
    .astype(np.int64)
)


impressions = pl.read_parquet(
    PROCESSED / f"impressions_{TARGET_SPLIT}.parquet",
    columns=["impression_uid", "timestamp"],
)
profile_parts = sorted(PROFILE_DIR.glob("*.parquet"))
profiles = pl.read_parquet(
    profile_parts,
    columns=["impression_uid", "timestamp", "full_history_rows"],
).sort(["timestamp", "impression_uid"])

missing = (
    impressions.select("impression_uid")
    .unique()
    .join(
        profiles.select("impression_uid"),
        on="impression_uid",
        how="anti",
    )
)
if missing.height:
    raise ValueError(f"{missing.height:,} impressions have no request profile")

uid = profiles["impression_uid"].to_numpy().astype(np.int64)
timestamps = profiles["timestamp"].to_numpy()
ts = profiles["timestamp"].dt.epoch("ns").to_numpy().astype(np.int64)
history_rows = [
    rows_ if rows_ else [] for rows_ in profiles["full_history_rows"].to_list()
]
hour = (ts - T0_NS) // HOUR_NS

print(f"Catalogue articles: {n_items - 1:,}")
print(f"Requests: {uid.size:,}")
print(f"Requests with history: {sum(bool(r) for r in history_rows):,}")

Catalogue articles: 51,282
Requests: 31,624
Requests with history: 31,510


## 3. Build the rolling popularity state

Aggregate the logged candidate slates by `(news_code, hour)` for every source split into dense exposure and click matrices. A request in hour `h` reads only row `h - 1`, so the popularity score is strictly causal.

In [3]:
SOURCE_SPLITS = (
    ["train", "validation"]
    if TARGET_SPLIT == "validation"
    else ["train", "validation", "test"]
)

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet").select(
                "news_code",
                "clicked",
                ((pl.col("timestamp").dt.epoch("ns") - T0_NS) // HOUR_NS).alias("hour"),
            )
            for split in SOURCE_SPLITS
        ]
    )
    .group_by("news_code", "hour")
    .agg(
        exp=pl.len(),
        clk=pl.col("clicked").cast(pl.Int64).sum(),
    )
    .collect()
)

source_rows = row_of_code[hourly["news_code"].to_numpy().astype(np.int64)]
hour_idx = hourly["hour"].to_numpy()
N_HOURS = int(hour_idx.max()) + 1
if hour.min() < 0 or hour.max() >= N_HOURS:
    raise ValueError("Target impressions fall outside the hourly state range")

EXPOSURES = np.zeros((N_HOURS, n_items), dtype=np.int32)
CLICKS = np.zeros((N_HOURS, n_items), dtype=np.int32)
EXPOSURES[hour_idx, source_rows] = hourly["exp"].to_numpy()
CLICKS[hour_idx, source_rows] = hourly["clk"].to_numpy()

print(f"Source candidate rows: {int(EXPOSURES.sum()):,}")
print(f"Hourly states: {N_HOURS:,}")
print(f"Articles exposed at least once: {int((EXPOSURES.sum(0) > 0).sum()):,}")

Source candidate rows: 4,621,015
Hourly states: 120
Articles exposed at least once: 16,978


## 4. Generate the history-filtered popular channel

For each request, rank available articles by prior-hour clicks, then prior-hour exposures, then `news_code`. Walk the ranking, skipping unavailable and history articles, and keep the first 200.

In [4]:
def score_snapshot(hour_):
    """Return prior-hour clicks and prior-hour exposures."""
    if hour_ == 0:
        zero = np.zeros(n_items, dtype=np.int32)
        return zero, zero
    return CLICKS[hour_ - 1], EXPOSURES[hour_ - 1]


def eligible_order(hour_):
    """Available article rows ordered by prior-hour clicks, exposures, news_code."""
    hour_end = T0_NS + (hour_ + 1) * HOUR_NS
    eligible = np.flatnonzero(avail_row <= hour_end)
    score, tiebreak = score_snapshot(hour_)
    return eligible[
        np.lexsort((row_to_code[eligible], -tiebreak[eligible], -score[eligible]))
    ]


max_history = max((len(r) for r in history_rows), default=0)
popular_pool_depth = POPULAR_TOP_N + max_history + 4_000
popular_rows = np.zeros((uid.size, POPULAR_TOP_N), dtype=np.int64)

starts = np.flatnonzero(np.r_[True, hour[1:] != hour[:-1]])
ends = np.r_[starts[1:], len(hour)]

for hour_value, start, end in zip(
    hour[starts].tolist(), starts.tolist(), ends.tolist()
):
    idx = np.arange(start, end, dtype=np.int64)
    ranked = eligible_order(hour_value)
    pool = ranked[: min(ranked.size, popular_pool_depth)]

    lengths = np.array([len(history_rows[i]) for i in idx], dtype=np.int64)
    if lengths.sum():
        read_rows = np.concatenate(
            [
                np.asarray(history_rows[i], dtype=np.int64)
                for i in idx
                if history_rows[i]
            ]
        )
        history_keys = (
            np.repeat(np.arange(idx.size, dtype=np.int64), lengths) * n_items
            + read_rows
        )
        in_history = np.isin(
            np.arange(idx.size, dtype=np.int64)[:, None] * n_items + pool[None, :],
            history_keys,
        )
    else:
        in_history = np.zeros((idx.size, pool.size), dtype=bool)

    available = avail_row[pool][None, :] <= ts[idx][:, None]
    keep = available & ~in_history
    take = keep & (np.cumsum(keep, axis=1) <= POPULAR_TOP_N)
    counts = take.sum(axis=1)
    full = counts == POPULAR_TOP_N
    popular_rows[idx[full]] = pool[np.nonzero(take[full])[1]].reshape(-1, POPULAR_TOP_N)

    # Rare fallback: scan the full ranking when the fixed pool was too shallow.
    for local in np.flatnonzero(~full):
        request = idx[local]
        history_set = set(history_rows[request])
        chosen = []
        for article_row in ranked:
            if (
                avail_row[article_row] <= ts[request]
                and int(article_row) not in history_set
            ):
                chosen.append(int(article_row))
                if len(chosen) == POPULAR_TOP_N:
                    break
        if len(chosen) != POPULAR_TOP_N:
            raise ValueError(
                f"Request {uid[request]} has fewer than {POPULAR_TOP_N} popular articles"
            )
        popular_rows[request] = chosen

if (popular_rows == 0).any():
    raise ValueError("The popular channel did not fill every request")

popular_codes = row_to_code[popular_rows]
print(f"Popular candidates: {popular_rows.shape[0]:,} requests x {POPULAR_TOP_N}")

Popular candidates: 31,624 requests x 200


## 5. Load content representations and pool request profiles

Article vectors are the frozen stage 3 text and entity representations. Request profiles are built from the same causal histories, mean-pooled over valid vectors and L2-normalised. Empty or zero histories trigger the popularity fallback.

In [5]:
text = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
entity = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")

counts = pl.read_parquet(
    PROCESSED / "news_catalogue.parquet",
    columns=["news_code", "exposures_train"],
)
popularity = counts.select(
    "news_code", pl.col("exposures_train").alias("popularity_score")
)

catalogue = index.join(availability, on="news_code", how="left").join(
    popularity, on="news_code", how="left"
)
ordered = catalogue.sort(["available_from", "news_code"])

article_rows_sorted = ordered["article_row"].to_numpy().astype(np.int64)
news_sorted = ordered["news_code"].to_numpy().astype(np.int64)
available_sorted = ordered["available_from"].to_numpy().astype("datetime64[ns]")
popularity_sorted = ordered["popularity_score"].fill_null(0).to_numpy().astype(np.int64)
text_sorted = np.asarray(text[article_rows_sorted], dtype=np.float32)
entity_sorted = np.asarray(entity[article_rows_sorted], dtype=np.float32)
text_article_valid = np.linalg.norm(text_sorted, axis=1) > 0
entity_article_valid = np.linalg.norm(entity_sorted, axis=1) > 0

pos_in_sorted = np.empty(n_items, dtype=np.int64)
pos_in_sorted[article_rows_sorted] = np.arange(len(article_rows_sorted), dtype=np.int64)
global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))

print(
    f"Text article vectors: {int(text_article_valid.sum()):,} / {len(article_rows_sorted):,}"
)
print(
    f"Entity article vectors: {int(entity_article_valid.sum()):,} / {len(article_rows_sorted):,}"
)

Text article vectors: 51,282 / 51,282
Entity article vectors: 44,265 / 51,282


In [6]:
def pool_profiles(histories, matrix, block=10_000):
    """Mean-pool valid article vectors per request and L2-normalise."""
    matrix = np.asarray(matrix)
    n = len(histories)
    dim = matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)

    for start in range(0, n, block):
        chunk = histories[start : start + block]
        lengths = np.fromiter(
            (len(rows_) for rows_ in chunk), dtype=np.int64, count=len(chunk)
        )
        parts = [np.asarray(rows_, dtype=np.int64) for rows_ in chunk if len(rows_)]
        if not parts:
            continue
        flat = np.concatenate(parts)
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12

        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        group_valid = groups[valid]
        vector_valid = vectors[valid]
        present_counts = np.bincount(group_valid, minlength=len(chunk))
        sums = np.empty((len(chunk), dim), dtype=np.float32)
        for column in range(dim):
            sums[:, column] = np.bincount(
                group_valid,
                weights=vector_valid[:, column],
                minlength=len(chunk),
            )
        present = present_counts > 0
        pooled = sums[present] / present_counts[present, None]
        norms = np.linalg.norm(pooled, axis=1)
        pooled /= np.where(norms > 0, norms, 1.0)[:, None]
        result[start : start + len(chunk)][present] = pooled

    return result


text_profiles = pool_profiles(history_rows, text)
entity_profiles = pool_profiles(history_rows, entity)
text_profile_valid = np.linalg.norm(text_profiles, axis=1) > 0
entity_profile_valid = np.linalg.norm(entity_profiles, axis=1) > 0

print(
    f"Valid text profiles: {int(text_profile_valid.sum()):,} / {len(text_profiles):,}"
)
print(
    f"Valid entity profiles: {int(entity_profile_valid.sum()):,} / {len(entity_profiles):,}"
)

Valid text profiles: 31,510 / 31,624
Valid entity profiles: 31,454 / 31,624


## 6. Generate the content channel and write the combined table

Each timestamp is scored as a batch over the eligible catalogue. The fused score is `0.5 * text_cosine + 0.5 * entity_cosine` when both sides are valid, otherwise the valid side is used alone. History and the already-selected popular articles are excluded before taking 250 content candidates.

In [7]:
def fuse_scores(text_scores, entity_scores, text_available, entity_available):
    """Equal-weight fusion with valid-side renormalisation."""
    fused = np.full(text_scores.shape, -np.inf, dtype=np.float32)
    both = text_available & entity_available
    only_text = text_available & ~entity_available
    only_entity = entity_available & ~text_available
    fused[both] = 0.5 * (text_scores[both] + entity_scores[both])
    fused[only_text] = text_scores[only_text]
    fused[only_entity] = entity_scores[only_entity]
    return fused


def popularity_select(order, n_eligible, excluded, k):
    """Take popular eligible positions from a precomputed global order."""
    excluded_set = set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position < n_eligible and position not in excluded_set:
            selected.append(position)
            if len(selected) == k:
                break
    return np.asarray(selected, dtype=np.int64)


def select_top_k(scores, popularity_, news_code, k, excluded, fallback_order=None):
    """Deterministic Top-k by score, popularity, then news_code."""
    n = scores.size
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded, k)

    positions = np.asarray(excluded, dtype=np.int64)
    if positions.size:
        scores = scores.copy()
        scores[positions] = -np.inf

    frontier = np.argpartition(-scores, k - 1)[:k]
    threshold = scores[frontier].min()
    above = np.flatnonzero(scores > threshold)
    need = k - above.size
    ties = np.flatnonzero(scores == threshold)
    if ties.size > need:
        tie_order = np.lexsort((news_code[ties], -popularity_[ties]))
        ties = ties[tie_order[:need]]
    chosen_positions = np.r_[above, ties]

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity_[chosen_positions],
            -scores[chosen_positions],
        )
    )
    return chosen_positions[order].astype(np.int64)

In [8]:
popular_positions = pos_in_sorted[popular_rows]
content_rows = np.zeros((uid.size, CONTENT_TOP_N), dtype=np.int64)

starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
batches = list(zip(starts.tolist(), ends.tolist()))

print(f"Scoring requests: {uid.size:,}")

for batch_no, (start, end) in enumerate(batches, 1):
    timestamp = timestamps[start]
    n_eligible = int(np.searchsorted(available_sorted, timestamp, side="right"))

    popularity_batch = popularity_sorted[:n_eligible]
    news_batch = news_sorted[:n_eligible]
    text_batch = text_sorted[:n_eligible]
    entity_batch = entity_sorted[:n_eligible]
    text_article_available = text_article_valid[:n_eligible]
    entity_article_available = entity_article_valid[:n_eligible]

    text_scores = text_profiles[start:end] @ text_batch.T
    entity_scores = entity_profiles[start:end] @ entity_batch.T

    text_available = (
        text_profile_valid[start:end, None] & text_article_available[None, :]
    )
    entity_available = (
        entity_profile_valid[start:end, None] & entity_article_available[None, :]
    )
    text_scores[~text_available] = -np.inf
    entity_scores[~entity_available] = -np.inf
    fused_scores = fuse_scores(
        text_scores, entity_scores, text_available, entity_available
    )

    for local, profile_row in enumerate(range(start, end)):
        excluded_parts = []

        history = history_rows[profile_row]
        if history:
            history_pos = pos_in_sorted[np.asarray(history, dtype=np.int64)]
            excluded_parts.append(history_pos[history_pos < n_eligible])

        popular_pos = popular_positions[profile_row]
        excluded_parts.append(popular_pos[popular_pos < n_eligible])
        excluded = (
            np.concatenate(excluded_parts)
            if excluded_parts
            else np.empty(0, dtype=np.int64)
        )

        fallback = (
            None
            if (text_profile_valid[profile_row] or entity_profile_valid[profile_row])
            else global_popularity_order
        )
        content_rows[profile_row] = select_top_k(
            fused_scores[local],
            popularity_batch,
            news_batch,
            CONTENT_TOP_N,
            excluded,
            fallback,
        )

    if batch_no % 5000 == 0 or batch_no == len(batches):
        print(f"Scored {batch_no:,} / {len(batches):,} timestamp batches", flush=True)

content_codes = news_sorted[content_rows]
combined_codes = np.concatenate([popular_codes, content_codes], axis=1)

if combined_codes.shape != (uid.size, CANDIDATE_CAP):
    raise ValueError("Combined candidate matrix has the wrong shape")
ordered_codes = np.sort(combined_codes, axis=1)
if (ordered_codes[:, 1:] == ordered_codes[:, :-1]).any():
    raise ValueError("Combined lists contain duplicate articles")

print(f"Popularity candidates: {popular_codes.shape[1]}")
print(f"Content candidates: {content_codes.shape[1]}")
print(f"Combined candidates per request: {CANDIDATE_CAP}")

Scoring requests: 31,624
Scored 5,000 / 24,580 timestamp batches
Scored 10,000 / 24,580 timestamp batches
Scored 15,000 / 24,580 timestamp batches
Scored 20,000 / 24,580 timestamp batches
Scored 24,580 / 24,580 timestamp batches
Popularity candidates: 200
Content candidates: 250
Combined candidates per request: 450


In [9]:
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
writer = CandidateWriter(OUTPUT_PATH)
try:
    for start in range(0, uid.size, 2_000):
        end = min(start + 2_000, uid.size)
        block = combined_codes[start:end]
        writer.add(
            np.repeat(uid[start:end], CANDIDATE_CAP),
            np.tile(cutoff_values, end - start),
            block.reshape(-1),
        )
finally:
    writer.close()

saved_rows = pl.scan_parquet(OUTPUT_PATH).select(pl.len()).collect().item()
expected_rows = uid.size * CANDIDATE_CAP
if saved_rows != expected_rows:
    raise ValueError(f"Expected {expected_rows:,} rows, found {saved_rows:,}")

print(f"Candidate rows: {saved_rows:,}")
print("Candidates saved to:", OUTPUT_PATH)

Candidate rows: 14,230,800
Candidates saved to: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\baseline_content_popularity_combined\recommendations_validation_top450.parquet


## 7. Evaluation

Uses the shared `CandidateEvaluator` and updates `results/candidate_generation_comparison_validation.csv`.

In [10]:
evaluator = CandidateEvaluator(root=DATA_ROOT, split=TARGET_SPLIT)
evaluation = evaluator.evaluate_to_csv(
    OUTPUT_PATH,
    model_name=MODEL_NAME,
    m_values=M_VALUES,
    output_dir=RESULTS,
)

print(
    "Comparison updated:",
    RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv",
)
print(
    "Evaluation saved to:",
    RESULTS / f"candidate_generation_{MODEL_NAME}_{TARGET_SPLIT}.csv",
)
print(wide_table(evaluation))

Comparison updated: C:\Users\Yong Rui\BT4222-Group-6-\results\candidate_generation_comparison_validation.csv
Evaluation saved to: C:\Users\Yong Rui\BT4222-Group-6-\results\candidate_generation_baseline_content_popularity_combined_validation.csv
shape: (1, 11)
┌────────────┬────────────┬─────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ model      ┆ split      ┆ m   ┆ catalogue_ ┆ … ┆ novelty_bi ┆ personalis ┆ recall_im ┆ recall_po │
│ ---        ┆ ---        ┆ --- ┆ coverage   ┆   ┆ ts         ┆ ation      ┆ pression_ ┆ oled      │
│ str        ┆ str        ┆ i64 ┆ ---        ┆   ┆ ---        ┆ ---        ┆ mean      ┆ ---       │
│            ┆            ┆     ┆ f64        ┆   ┆ f64        ┆ f64        ┆ ---       ┆ f64       │
│            ┆            ┆     ┆            ┆   ┆            ┆            ┆ f64       ┆           │
╞════════════╪════════════╪═════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ baseline_c ┆ validation ┆ 450 ┆

## Notes and limitations

- The notebook is standalone with respect to the two baseline notebooks: it rebuilds the popularity order and the content scores from stage 1, 3 and 4 artifacts. It does not read `artifacts/popularity_freshness_prior_hour_clicks` or `artifacts/content_text_entity`.
- The combined table is popularity-first: `cutoff = 1..200` is the history-filtered rolling popularity channel, and `cutoff = 201..450` is the history-filtered content channel. Recall@M for `M <= 200` therefore reflects popularity-only performance.
- Every article in `full_history_rows` is removed from both channels before selection, so repeat-click opportunities are deliberately excluded.
- Popularity uses only completed hours strictly before the request hour. The hourly counts are built from the logged candidate slates, so they include realised click labels only for earlier hours.
- Content ties are broken by training popularity descending, then `news_code` ascending, matching the content baseline.
- `available_from` is the project-wide observed-availability proxy, not a verified publication timestamp.
- The implementation currently targets `validation` or `test`. For `test`, all three source splits are used for the hourly popularity state.